# APJN Colab Demo

This notebook clones the repo, runs permutation-symmetric input APJN experiments, runs CIFAR backward/forward APJN measurements with saved activation statistics, and compares theory to experiment.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/placeholder-user/norm-free-transformer-apjn-refactor.git"
REPO_DIR = Path("norm-free-transformer-apjn-refactor")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm", "matplotlib"], check=True)
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
if str(REPO_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(REPO_DIR.resolve()))

print("Working directory:", Path.cwd())

In [ ]:
import matplotlib.pyplot as plt

from apjn import (
    ModelConfig,
    PermSymInputExpConfig,
    build_backward_theory_comparison,
    build_forward_theory_comparison,
    build_perm_inv_input_exp_theory_comparison,
    compute_depthwise_gmfe_records,
    load_saved_bundle,
    run_cifar_backward_apjn_with_activation_stats,
    run_cifar_forward_apjn_with_activation_stats,
    run_perm_inv_input_apjn_exp,
)
from vit_apjn_plots import (
    plot_depthwise_gmfe,
    plot_perm_inv_input_apjn_comparison,
    plot_theory_vs_experiment_curves,
)

MODEL_CFG = ModelConfig(
    model_name="vit_base_patch16_224",
    depth=64,
    num_classes=100,
    img_size=224,
    seed=124,
)
DEFAULT_ALPHAS = (0.5, 0.8, 1.0)
ATTN_MULT = 1.0
MLP_MULT = 1.0

## 1. Permutation-symmetric inputs

In [ ]:
PERM_CFG = PermSymInputExpConfig(
    batch_size=16,
    equiangular_q0=1.0,
    equiangular_p0=0.2,
    equiangular_seed=0,
    equiangular_random_rotate=True,
    backward_apjn_layers=tuple(range(4, MODEL_CFG.depth, 4)),
    forward_apjn_layers=tuple(range(4, MODEL_CFG.depth, 4)),
    forward_source_block=4,
    alphas=DEFAULT_ALPHAS,
    j_num_draws=10,
    num_model_inits=4,
    attn_mult=ATTN_MULT,
    mlp_mult=MLP_MULT,
)

perm_out = run_perm_inv_input_apjn_exp(MODEL_CFG, PERM_CFG)
perm_comparison = build_perm_inv_input_exp_theory_comparison(perm_out)
fig, _ = plot_perm_inv_input_apjn_comparison(perm_comparison, figsize=(14, 5))
plt.show()

## 2. Backward APJNs on CIFAR-100

In [ ]:
ACTIVATION_STAT_BLOCKS = (0, 4, 8, 16, 32, 40)
SEED = 124
NUM_MODEL_INITS = 8
N_SAMPLES = 24
BATCH_SIZE = 24
CIFAR_LAYER_STRIDE = 4
POSTFIX = (
    f"determ_final_more_batch_{BATCH_SIZE}_seed_{SEED}_"
    f"num_inits_{NUM_MODEL_INITS}_attn_mult_{ATTN_MULT}_mlp_mult_{MLP_MULT}"
)

backward_out = run_cifar_backward_apjn_with_activation_stats(
    MODEL_CFG,
    alphas=DEFAULT_ALPHAS,
    n_samples=N_SAMPLES,
    layer_stride=CIFAR_LAYER_STRIDE,
    batch_size=BATCH_SIZE,
    batch_seed=SEED,
    j_num_draws=10,
    activation_stat_blocks=ACTIVATION_STAT_BLOCKS,
    num_model_inits=NUM_MODEL_INITS,
    attn_mult=ATTN_MULT,
    mlp_mult=MLP_MULT,
    deterministic=True,
    save_results=True,
    save_root="/content/apjn_results",
    result_postfix=POSTFIX,
    keep_pre_blocks_init=False,
    skip_preln=False,
)
backward_saved = load_saved_bundle(backward_out["saved_path"])
print(backward_out["saved_path"])

In [ ]:
BLOCK_INIT_CONDITION = 0
backward_comparisons = [
    build_backward_theory_comparison(
        backward_saved,
        sample_index=0,
        block_init_condition=BLOCK_INIT_CONDITION,
        model_variant="preln",
    )
]
backward_comparisons.extend(
    build_backward_theory_comparison(
        backward_saved,
        sample_index=0,
        block_init_condition=BLOCK_INIT_CONDITION,
        model_variant="derf",
        alpha=alpha,
    )
    for alpha in DEFAULT_ALPHAS
)

fig, _ = plot_theory_vs_experiment_curves(backward_comparisons, direction="backward", figsize=(9, 5))
plt.show()

In [ ]:
backward_gmfe = compute_depthwise_gmfe_records(
    backward_saved,
    direction="backward",
    block_init_condition=0,
    block_start_comparing_apjn=4,
    alphas=DEFAULT_ALPHAS,
)
fig, _ = plot_depthwise_gmfe(backward_gmfe, metric_key="typ_mult_err", figsize=(10, 5))
plt.show()

## 3. Forward APJNs on CIFAR-100

In [ ]:
FORWARD_SOURCE_BLOCK = 4

forward_out = run_cifar_forward_apjn_with_activation_stats(
    MODEL_CFG,
    alphas=DEFAULT_ALPHAS,
    source_block_index=FORWARD_SOURCE_BLOCK,
    n_samples=N_SAMPLES,
    layer_stride=CIFAR_LAYER_STRIDE,
    batch_size=BATCH_SIZE,
    batch_seed=SEED,
    j_num_draws=10,
    activation_stat_blocks=ACTIVATION_STAT_BLOCKS,
    num_model_inits=NUM_MODEL_INITS,
    attn_mult=ATTN_MULT,
    mlp_mult=MLP_MULT,
    deterministic=True,
    save_results=True,
    save_root="/content/apjn_results",
    result_postfix=POSTFIX + f"_forward_src_{FORWARD_SOURCE_BLOCK}",
    keep_pre_blocks_init=False,
    skip_preln=False,
)
forward_saved = load_saved_bundle(forward_out["saved_path"])
print(forward_out["saved_path"])

In [ ]:
forward_comparisons = [
    build_forward_theory_comparison(
        forward_saved,
        sample_index=0,
        block_init_condition=FORWARD_SOURCE_BLOCK,
        model_variant="preln",
    )
]
forward_comparisons.extend(
    build_forward_theory_comparison(
        forward_saved,
        sample_index=0,
        block_init_condition=FORWARD_SOURCE_BLOCK,
        model_variant="derf",
        alpha=alpha,
    )
    for alpha in DEFAULT_ALPHAS
)

fig, _ = plot_theory_vs_experiment_curves(forward_comparisons, direction="forward", figsize=(9, 5))
plt.show()

In [ ]:
forward_gmfe = compute_depthwise_gmfe_records(
    forward_saved,
    direction="forward",
    block_init_condition=FORWARD_SOURCE_BLOCK,
    block_start_comparing_apjn=FORWARD_SOURCE_BLOCK,
    alphas=DEFAULT_ALPHAS,
)
fig, _ = plot_depthwise_gmfe(forward_gmfe, metric_key="typ_mult_err", figsize=(10, 5))
plt.show()